# TPF Zooniverse Subject Videos

Experiment with animated TESS target-pixel flux-difference videos for Zooniverse subjects or candidate UUIDs. The animation uses the same subject/candidate lookup, automatic peak-centered zoom window, and pre-event TPF reference-frame approach as the static TPF review images.

In [ ]:
from pathlib import Path

import numpy
from IPython.display import HTML, display
from matplotlib import pyplot
from matplotlib.animation import FuncAnimation
from matplotlib.colors import Normalize
from matplotlib.ticker import FormatStrFormatter

from searchlight.lightcurves import load
from searchlight.public_candidate_exports import (
    DEFAULT_TPF_DOWNLOAD_DIR,
    TPF_DIFF_LIMIT_PERCENTILE,
    TPF_MIN_ABS_DIFF_LIMIT,
    TPF_REFERENCE_LOOKBACK_DAYS,
    download_tess_tpf,
    extract_tess_tpf_metadata,
    flux_to_numpy,
    nanmean_frame,
    resolve_zoom_window,
)
from searchlight.review_exports import (
    DEFAULT_METADATA_CACHE_PATH,
    DEFAULT_RESULT_CACHE_META_PATH,
    DEFAULT_RESULT_CACHE_PATH,
    enrich_results_with_metadata,
    load_lensing_results,
)
from searchlight.zooniverse_results import (
    DEFAULT_SUBJECT_METADATA_CACHE_PATH,
    add_candidate_ids,
    select_zooniverse_subject_results,
)


In [ ]:
RESULTS_PATH = Path('data/results/simulens1.0_tess_spoc')
RESULT_CACHE_PATH = DEFAULT_RESULT_CACHE_PATH
RESULT_CACHE_META_PATH = DEFAULT_RESULT_CACHE_META_PATH
RESULT_METADATA_CACHE_PATH = DEFAULT_METADATA_CACHE_PATH

SUBJECT_METADATA_CACHE_PATH = DEFAULT_SUBJECT_METADATA_CACHE_PATH
TPF_DOWNLOAD_DIR = DEFAULT_TPF_DOWNLOAD_DIR

# Add Zooniverse subject IDs or candidate UUIDs here.
# Numeric IDs are looked up through Zooniverse; UUIDs are matched directly.
REVIEW_IDS = [119505889, 119507228, 119514070, 119517568, 119529883, 119607748, 119533173]
# Set to an integer for a quick preview; None renders every requested ID.
TOP_N = None

# Leave these as None to use the automatic peak-centered zoom window.
# Use relative days from the first cadence, for example (12.4, 13.1).
ZOOM_WINDOW_DAYS = None
# Or use absolute JD timestamps, for example (2459800.4, 2459801.1).
ZOOM_WINDOW_JD = None

# Optional per-subject overrides when reviewing several subjects.
# Example: {119539666: (12.4, 13.1)}
ZOOM_WINDOWS_DAYS_BY_SUBJECT_ID = {}
ZOOM_WINDOWS_JD_BY_SUBJECT_ID = {}

# Animation controls. Lower MAX_VIDEO_FRAMES if HTML rendering gets slow.
VIDEO_FPS = 8
MAX_VIDEO_FRAMES = 160
REFERENCE_LOOKBACK_DAYS = TPF_REFERENCE_LOOKBACK_DAYS


In [ ]:
results = load_lensing_results(
    results_path=RESULTS_PATH,
    cache_path=RESULT_CACHE_PATH,
    cache_meta_path=RESULT_CACHE_META_PATH,
)
results = add_candidate_ids(results)
print(f'Loaded {len(results)} lensing model rows from {RESULTS_PATH}')


In [ ]:
selected = select_zooniverse_subject_results(
    results,
    REVIEW_IDS,
    subject_metadata_cache_path=SUBJECT_METADATA_CACHE_PATH,
)
review_results = selected.copy()
if TOP_N is not None:
    review_results = review_results.head(TOP_N).copy()
review_results[['subject_id', 'candidate_id', 'path', 'lensing']].head()


In [ ]:
review_export = enrich_results_with_metadata(
    review_results,
    cache_path=RESULT_METADATA_CACHE_PATH,
)
review_export[[
    'subject_id',
    'candidate_id',
    'path',
    'lensing',
    'tic',
    'sector',
    'simbad_url',
]].head()


In [ ]:
def _subject_zoom_window(mapping, subject_id):
    return mapping.get(subject_id, mapping.get(str(subject_id)))


def normalized_lightcurve(path):
    lc = load(path)
    absolute_time_jd = numpy.asarray(lc['time'].jd, dtype=float)
    time_zero_jd = absolute_time_jd[0]
    time_days = absolute_time_jd - time_zero_jd
    brightness = flux_to_numpy(lc['flux'])

    finite = numpy.isfinite(time_days) & numpy.isfinite(brightness)
    time_days = time_days[finite]
    brightness = brightness[finite]

    if len(time_days) == 0:
        raise ValueError(f'No finite data points found in {path}')

    median_brightness = numpy.nanmedian(brightness)
    if numpy.isfinite(median_brightness) and median_brightness != 0:
        brightness = brightness / median_brightness

    return lc, time_zero_jd, time_days, brightness, median_brightness


def build_tpf_video_sequences(
    tpf,
    zoom_window,
    time_zero_jd,
    lightcurve_median_flux,
    reference_lookback_days=REFERENCE_LOOKBACK_DAYS,
):
    if not numpy.isfinite(lightcurve_median_flux) or lightcurve_median_flux == 0:
        raise ValueError('Light-curve median flux must be finite and non-zero')

    flux_cube = flux_to_numpy(tpf.flux)
    if flux_cube.ndim != 3 or len(flux_cube) == 0:
        raise ValueError('Unexpected target pixel file shape')

    tpf_time_jd = numpy.asarray(tpf.time.jd, dtype=float)
    finite_frames = numpy.flatnonzero(numpy.isfinite(tpf_time_jd))
    if len(finite_frames) == 0:
        raise ValueError('No finite target pixel frames available')

    window_start_jd = zoom_window['start_jd']
    window_end_jd = zoom_window['end_jd']
    reference_start_jd = window_start_jd - reference_lookback_days

    window_frames = finite_frames[
        (tpf_time_jd[finite_frames] >= window_start_jd)
        & (tpf_time_jd[finite_frames] <= window_end_jd)
    ]
    reference_frames = finite_frames[
        (tpf_time_jd[finite_frames] >= reference_start_jd)
        & (tpf_time_jd[finite_frames] < window_start_jd)
    ]
    if len(window_frames) == 0:
        raise ValueError('No target pixel frames fall inside the zoom window')
    if len(reference_frames) == 0:
        reference_frames = finite_frames[tpf_time_jd[finite_frames] < window_start_jd]
    if len(reference_frames) == 0:
        print('No pre-window TPF frames; using the first in-window frame as the reference.')
        reference_frames = window_frames[:1]

    reference_frame = nanmean_frame(
        [numpy.array(flux_cube[index], dtype=float, copy=True) for index in reference_frames]
    )

    relative_days = tpf_time_jd[window_frames] - time_zero_jd
    raw_frames = numpy.array(
        [numpy.array(flux_cube[index], dtype=float, copy=True) for index in window_frames]
    )
    difference_frames = numpy.array(
        [
            (raw_frame - reference_frame) / lightcurve_median_flux
            for raw_frame in raw_frames
        ]
    )
    raw_values = raw_frames[numpy.isfinite(raw_frames)]
    if len(raw_values) == 0:
        raise ValueError('No finite target pixel flux values available')

    finite_difference = difference_frames[numpy.isfinite(difference_frames)]
    if len(finite_difference) == 0:
        raise ValueError('No finite target pixel difference values available')

    abs_limit = numpy.nanpercentile(
        numpy.abs(finite_difference),
        TPF_DIFF_LIMIT_PERCENTILE,
    )
    if not numpy.isfinite(abs_limit) or abs_limit == 0:
        abs_limit = numpy.nanmax(numpy.abs(finite_difference))
    if not numpy.isfinite(abs_limit) or abs_limit == 0:
        abs_limit = 1.0
    abs_limit = max(abs_limit, TPF_MIN_ABS_DIFF_LIMIT)

    raw_vmin, raw_vmax = numpy.nanpercentile(raw_values, [2, 98])
    if not numpy.isfinite(raw_vmin) or not numpy.isfinite(raw_vmax) or raw_vmin == raw_vmax:
        raw_vmin = numpy.nanmin(raw_values)
        raw_vmax = numpy.nanmax(raw_values)
    if not numpy.isfinite(raw_vmin) or not numpy.isfinite(raw_vmax) or raw_vmin == raw_vmax:
        raw_vmin, raw_vmax = 0.0, 1.0

    return (
        relative_days,
        raw_frames,
        difference_frames,
        Normalize(vmin=raw_vmin, vmax=raw_vmax),
        Normalize(vmin=-abs_limit, vmax=abs_limit),
    )


def thin_video_frames(relative_days, raw_frames, difference_frames, max_frames=MAX_VIDEO_FRAMES):
    if max_frames is None or len(relative_days) <= max_frames:
        return relative_days, raw_frames, difference_frames
    indices = numpy.linspace(0, len(relative_days) - 1, max_frames).round().astype(int)
    indices = numpy.unique(indices)
    return relative_days[indices], raw_frames[indices], difference_frames[indices]


def make_tpf_difference_video(
    row,
    zoom_window_days=None,
    zoom_window_jd=None,
    tpf_download_dir=TPF_DOWNLOAD_DIR,
    fps=VIDEO_FPS,
    max_frames=MAX_VIDEO_FRAMES,
):
    path = row['path']
    lc, time_zero_jd, time_days, brightness, median_brightness = normalized_lightcurve(path)
    zoom_window = resolve_zoom_window(
        time_days,
        brightness,
        time_zero_jd,
        zoom_window_days=zoom_window_days,
        zoom_window_jd=zoom_window_jd,
    )

    ticid, sector, ra, dec = extract_tess_tpf_metadata(path)
    tpf = download_tess_tpf(
        ticid=ticid,
        sector=sector,
        ra=ra,
        dec=dec,
        download_dir=tpf_download_dir,
    )
    frame_days, raw_frames, difference_frames, raw_norm, difference_norm = build_tpf_video_sequences(
        tpf,
        zoom_window,
        time_zero_jd,
        median_brightness,
    )
    frame_days, raw_frames, difference_frames = thin_video_frames(
        frame_days,
        raw_frames,
        difference_frames,
        max_frames,
    )

    x_left = zoom_window['start_day']
    x_right = zoom_window['end_day']
    in_window = (time_days >= x_left) & (time_days <= x_right)

    fig = pyplot.figure(
        figsize=(10.5, 8.5),
        constrained_layout=True,
    )
    grid = fig.add_gridspec(
        3,
        2,
        height_ratios=[1.35, 1.0, 0.72],
        width_ratios=[1.0, 1.0],
    )
    ax_raw = fig.add_subplot(grid[0, 0])
    ax_tpf = fig.add_subplot(grid[0, 1])
    ax_lc = fig.add_subplot(grid[1, :])
    ax_full_lc = fig.add_subplot(grid[2, :])
    fig.patch.set_facecolor('#f7f4ea')

    raw_image = ax_raw.imshow(
        raw_frames[0],
        origin='lower',
        cmap='viridis',
        norm=raw_norm,
        animated=True,
    )
    ax_raw.set_title('Original TPF flux', loc='left', fontsize=11)
    ax_raw.set_xticks([])
    ax_raw.set_yticks([])
    raw_colorbar = fig.colorbar(raw_image, ax=ax_raw, fraction=0.046, pad=0.04)
    raw_colorbar.set_label('Flux')

    difference_image = ax_tpf.imshow(
        difference_frames[0],
        origin='lower',
        cmap='RdBu_r',
        norm=difference_norm,
        animated=True,
    )
    ax_tpf.set_title('TPF flux difference from pre-window reference', loc='left', fontsize=11)
    ax_tpf.set_xticks([])
    ax_tpf.set_yticks([])
    difference_colorbar = fig.colorbar(difference_image, ax=ax_tpf, fraction=0.046, pad=0.04)
    difference_colorbar.set_label('Brightness difference')

    ax_lc.scatter(
        time_days[in_window],
        brightness[in_window],
        s=11,
        color='#24445c',
        alpha=0.75,
        edgecolors='none',
    )
    marker_line = ax_lc.axvline(frame_days[0], color='#b44a2f', linewidth=1.7)
    marker_point, = ax_lc.plot([], [], 'o', color='#b44a2f', markersize=5)
    ax_lc.set_xlim(x_left, x_right)
    if in_window.any():
        y_min = float(numpy.nanmin(brightness[in_window]))
        y_max = float(numpy.nanmax(brightness[in_window]))
        if y_min < y_max:
            y_pad = max((y_max - y_min) * 0.12, 0.005)
            ax_lc.set_ylim(y_min - y_pad, y_max + y_pad)
    ax_lc.grid(axis='y', color='#d8dfdf', linewidth=0.8, alpha=0.8)
    ax_lc.set_xlabel('Time (days)')
    ax_lc.set_ylabel('Brightness')
    ax_lc.set_title('Zoomed light curve', loc='left', fontsize=11)
    ax_lc.xaxis.set_major_formatter(FormatStrFormatter('%.2f'))
    time_label = ax_lc.text(
        0.02,
        0.96,
        '',
        transform=ax_lc.transAxes,
        ha='left',
        va='top',
        bbox={'facecolor': '#fffdf8', 'edgecolor': '#d6cfc2', 'alpha': 0.95},
    )

    ax_full_lc.scatter(
        time_days,
        brightness,
        s=4,
        color='#24445c',
        alpha=0.5,
        edgecolors='none',
    )
    ax_full_lc.axvspan(x_left, x_right, color='#d7c5a5', alpha=0.42, linewidth=0)
    full_marker_line = ax_full_lc.axvline(frame_days[0], color='#b44a2f', linewidth=1.2)
    finite_all = numpy.isfinite(time_days) & numpy.isfinite(brightness)
    if finite_all.any():
        ax_full_lc.set_xlim(float(numpy.nanmin(time_days[finite_all])), float(numpy.nanmax(time_days[finite_all])))
        y_min = float(numpy.nanmin(brightness[finite_all]))
        y_max = float(numpy.nanmax(brightness[finite_all]))
        if y_min < y_max:
            y_pad = max((y_max - y_min) * 0.08, 0.005)
            ax_full_lc.set_ylim(y_min - y_pad, y_max + y_pad)
    ax_full_lc.grid(axis='y', color='#d8dfdf', linewidth=0.8, alpha=0.65)
    ax_full_lc.set_xlabel('Time (days)')
    ax_full_lc.set_ylabel('Brightness')
    ax_full_lc.set_title('Full light curve', loc='left', fontsize=11)
    ax_full_lc.xaxis.set_major_formatter(FormatStrFormatter('%.1f'))

    for ax in (ax_raw, ax_tpf, ax_lc, ax_full_lc):
        ax.set_facecolor('#fffdf8')
        for spine in ax.spines.values():
            spine.set_color('#9aa8b6')

    def update(frame_index):
        frame_day = frame_days[frame_index]
        raw_image.set_array(raw_frames[frame_index])
        difference_image.set_array(difference_frames[frame_index])
        marker_line.set_xdata([frame_day, frame_day])
        full_marker_line.set_xdata([frame_day, frame_day])
        nearest_lc_index = int(numpy.nanargmin(numpy.abs(time_days - frame_day)))
        marker_point.set_data([time_days[nearest_lc_index]], [brightness[nearest_lc_index]])
        time_label.set_text(f'{frame_day:.4f} days since first cadence')
        return raw_image, difference_image, marker_line, full_marker_line, marker_point, time_label

    animation = FuncAnimation(
        fig,
        update,
        frames=len(frame_days),
        interval=1000 / fps,
        blit=True,
    )
    pyplot.close(fig)
    animation.searchlight_zoom_window = zoom_window
    animation.searchlight_frame_count = len(frame_days)
    return animation


In [ ]:
for row in review_results.to_dict('records'):
    subject_id = row.get('subject_id')
    zoom_window_days = _subject_zoom_window(
        ZOOM_WINDOWS_DAYS_BY_SUBJECT_ID,
        subject_id,
    ) or ZOOM_WINDOW_DAYS
    zoom_window_jd = _subject_zoom_window(
        ZOOM_WINDOWS_JD_BY_SUBJECT_ID,
        subject_id,
    ) or ZOOM_WINDOW_JD

    animation = make_tpf_difference_video(
        row,
        zoom_window_days=zoom_window_days,
        zoom_window_jd=zoom_window_jd,
    )
    zoom_window = animation.searchlight_zoom_window
    display(HTML(
        '<div>'
        f'<strong>candidate_id:</strong> <code>{row["candidate_id"]}</code><br>'
        f'<strong>subject_id:</strong> <code>{subject_id}</code><br>'
        f'<strong>zoom:</strong> <code>{zoom_window["start_day"]:.6f}</code> to '
        f'<code>{zoom_window["end_day"]:.6f}</code> days ({zoom_window["source"]})<br>'
        f'<strong>frames:</strong> {animation.searchlight_frame_count}'
        '</div>'
    ))
    display(HTML(animation.to_jshtml()))
